# 🔬 HAM10000 Production Model Training: EfficientNet-B4 + Zero-Leakage + Calibration + Dual-Output ONNX
### DermAssist AI Medical Core Training Engine
---
**Platform**: Kaggle GPU (NVIDIA Tesla T4 x2 or P100)
**Dataset**: `kmader/skin-cancer-mnist-ham10000`

### 🎯 Production Invariants Guaranteed:
1. **Zero Patient-Level Data Leakage**: Lesions grouped strictly by `lesion_id` using `StratifiedGroupKFold`.
2. **Class Imbalance Mitigation**: Class-weighted Cross-Entropy addressing 67% Melanocytic Nevus (`nv`) prevalence.
3. **Compound Scaling**: EfficientNet-B4 operating at 380x380 resolution.
4. **Post-Hoc Temperature Calibration**: Optimization of $T^*$ via validation NLL to minimize Expected Calibration Error (ECE).
5. **Dual-Output ONNX Export**: Emits both classification `logits` (7,) and convolutional `features` (1792, H, W) for live Grad-CAM explainability.


In [ ]:
# 1. Environment Setup and Library Imports
import os
import sys
import glob
import json
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score
from scipy.optimize import minimize_scalar

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Verify GPU hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')
if torch.cuda.is_available():
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'Device Count: {torch.cuda.device_count()}')


In [ ]:
# 2. Configuration & Class Taxonomy
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = (380, 380) # EfficientNet-B4 native resolution
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4


In [ ]:
# 3. Dataset Discovery & Indexing
kaggle_base = '/kaggle/input/skin-cancer-mnist-ham10000'
csv_path = os.path.join(kaggle_base, 'HAM10000_metadata.csv')
image_dirs = [
    os.path.join(kaggle_base, 'HAM10000_images_part_1'),
    os.path.join(kaggle_base, 'HAM10000_images_part_2'),
]

df = pd.read_csv(csv_path)
print(f'Total raw metadata records: {len(df)}')

# Map image IDs to file paths
image_path_map = {}
for img_dir in image_dirs:
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        for file_path in glob.glob(os.path.join(img_dir, ext)):
            image_id = os.path.splitext(os.path.basename(file_path))[0]
            image_path_map[image_id] = file_path

df['file_path'] = df['image_id'].map(image_path_map)
df = df.dropna(subset=['file_path']).copy()
df['label'] = df['dx'].map(CLASS_TO_IDX)
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)

print(f"Indexed {len(df)} verified images across {df['lesion_id'].nunique()} unique lesions.")
df.head()


In [ ]:
# 4. Zero Patient Leakage Splitting via StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in sgkf.split(df, df['label'], groups=df['lesion_id']):
    train_df = df.iloc[train_idx].copy().reset_index(drop=True)
    val_df = df.iloc[val_idx].copy().reset_index(drop=True)
    break

# Mathematically assert zero lesion overlap
train_lesions = set(train_df['lesion_id'].unique())
val_lesions = set(val_df['lesion_id'].unique())
overlap = train_lesions.intersection(val_lesions)
assert len(overlap) == 0, f'CRITICAL LEAKAGE DETECTED: {len(overlap)} lesions overlap!'
print('✅ Zero Leakage Confirmed!')
print(f'Train Set: {len(train_df)} images ({len(train_lesions)} lesions)')
print(f'Val Set:   {len(val_df)} images ({len(val_lesions)} lesions)')


In [ ]:
# 5. Class Distribution & Balanced Loss Weight Computation
class_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)
raw_weights = total_samples / (len(CLASS_NAMES) * class_counts.astype(np.float32))
# Square-root damping to prevent gradient explosion on rare classes
damped_weights = np.sqrt(raw_weights)
damped_weights = damped_weights / damped_weights.sum() * len(CLASS_NAMES)
weight_tensor = torch.tensor(damped_weights, dtype=torch.float32).to(device)

for name, count, w in zip(CLASS_NAMES, class_counts, damped_weights):
    print(f'Class {name:6s} | Count: {count:5d} ({count/total_samples*100:5.2f}%) | Weight: {w:.3f}')


In [ ]:
# 6. PyTorch Dataset & DataLoaders
class HAM10000Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df
        self.transform = transform
        self.file_paths = df['file_path'].values
        self.labels = df['label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.file_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset = HAM10000Dataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# 7. Dual-Output EfficientNet-B4 Model Architecture
class DualOutputEfficientNetB4(nn.Module):
    def __init__(self, num_classes=7, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_b4(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features=in_features, out_features=num_classes)
        )

    def forward(self, x):
        features = self.backbone.features(x)
        pooled = self.backbone.avgpool(features)
        flattened = torch.flatten(pooled, 1)
        logits = self.backbone.classifier(flattened)
        return logits, features

model = DualOutputEfficientNetB4(num_classes=7, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [ ]:
# 8. Training & Validation Execution Loop
best_macro_f1 = 0.0
best_weights_path = 'best_model.pt'

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    running_loss, train_correct, total_train = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, _ = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        train_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
        total_train += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total_train
    train_acc = train_correct / total_train

    # Validation
    model.eval()
    val_loss = 0.0
    all_val_logits, all_val_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits, _ = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            all_val_logits.append(logits.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_dataset)
    val_logits_np = np.concatenate(all_val_logits, axis=0)
    val_labels_np = np.array(all_val_labels)
    val_preds_np = np.argmax(val_logits_np, axis=1)
    val_acc = np.mean(val_preds_np == val_labels_np)
    val_balanced_acc = balanced_accuracy_score(val_labels_np, val_preds_np)
    val_macro_f1 = f1_score(val_labels_np, val_preds_np, average='macro', zero_division=0)
    elapsed = time.time() - start_time

    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | ' 
          f'Val Loss: {val_loss:.4f} Acc: {val_acc*100:.1f}% BalAcc: {val_balanced_acc*100:.1f}% Macro-F1: {val_macro_f1:.4f}')

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        torch.save(model.state_dict(), best_weights_path)
        print(f'  --> 🌟 Checkpoint Saved (Macro-F1: {best_macro_f1:.4f})')


In [ ]:
# 9. Temperature Calibration Optimization (ECE Minimization)
model.load_state_dict(torch.load(best_weights_path, map_location=device))
model.eval()

final_val_logits, final_val_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits, _ = model(images)
        final_val_logits.append(logits.cpu().numpy())
        final_val_labels.extend(labels.numpy())

val_logits_np = np.concatenate(final_val_logits, axis=0)
val_labels_np = np.array(final_val_labels)

def softmax_np(x):
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True)

def compute_ece(probs, labels, n_bins=10):
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs = (preds == labels)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confs > bins[i]) & (confs <= bins[i+1])
        if np.sum(mask) > 0:
            ece += (np.sum(mask) / len(labels)) * np.abs(np.mean(accs[mask]) - np.mean(confs[mask]))
    return float(ece)

def nll_func(T):
    scaled = val_logits_np / max(0.05, float(T))
    probs = np.clip(softmax_np(scaled), 1e-12, 1.0 - 1e-12)
    return float(-np.sum(np.log(probs[np.arange(len(probs)), val_labels_np])) / len(probs))

raw_ece = compute_ece(softmax_np(val_logits_np), val_labels_np)
res = minimize_scalar(nll_func, bounds=(0.1, 10.0), method='bounded')
optimal_t = float(res.x)
calibrated_ece = compute_ece(softmax_np(val_logits_np / optimal_t), val_labels_np)

print(f'Uncalibrated Raw ECE: {raw_ece*100:.2f}%')
print(f'Calibrated ECE:       {calibrated_ece*100:.2f}%')
print(f'Optimal Temperature:  T* = {optimal_t:.4f}')


In [ ]:
# 10. Clinical Classification Report
val_preds_np = np.argmax(val_logits_np, axis=1)
print('\n--- CLINICAL VALIDATION CLASSIFICATION REPORT ---')
print(classification_report(val_labels_np, val_preds_np, target_names=CLASS_NAMES, digits=4))


In [ ]:
# 11. Dual-Output ONNX Export
model.cpu()
model.eval()
dummy_input = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1], dtype=torch.float32)
onnx_path = 'efficientnet_b4_ham10000.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits', 'features'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
        'features': {0: 'batch_size'}
    }
)
print(f'✅ Successfully exported {onnx_path} ({os.path.getsize(onnx_path)/(1024*1024):.2f} MB)')

# Save calibration parameters JSON
meta = {
    'model_architecture': 'EfficientNet-B4',
    'optimal_temperature': round(optimal_t, 4),
    'raw_ece': round(raw_ece, 4),
    'calibrated_ece': round(calibrated_ece, 4),
    'val_macro_f1': round(best_macro_f1, 4),
    'class_names': CLASS_NAMES
}
with open('calibration_params.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ Saved calibration_params.json')


## 📥 Next Step: Download & Deployment
From the Kaggle `/kaggle/working/` file browser in the right panel:
1. Download `efficientnet_b4_ham10000.onnx` and place in your project at `checkpoints/efficientnet_b4_ham10000.onnx`.
2. (Optional) Download `calibration_params.json`.
Your DermAssist AI web application is immediately ready with live inference and Grad-CAM explainability!
